# Final Risk Aggregator POC

This notebook merges the results from the two previous pipelines:
1. **Base Object & Quality Checks** (`outputs/reports/report.json`)
2. **Extra Person & Gaze Checks** (`outputs/reports/full_risk_report.csv`)

It applies a **Category-based Escalation Logic** to determine a final, combined risk score.

### Risk Categories
- **Category A (CRITICAL)**: Auto HIGH (e.g., Phone detected)
- **Category B (SUSPICIOUS)**: HIGH if 2+ present, or 1 combined with C. MEDIUM if alone. (e.g., Looking at extra person, Multiple faces)
- **Category C (CONTEXTUAL)**: MEDIUM if 2+ present. LOW if alone. (e.g., Book detected, Face missing)
- **Category D (AMBIENT)**: LOW. (e.g., Blur, Low light, Extra person far in background)


In [ ]:
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt

print("Imports OK")

## 1. Load the Data
We load the JSON from the first notebook and the CSV from the second notebook.

In [ ]:
report_json_path = Path("outputs/reports/report.json")
risk_csv_path    = Path("outputs/reports/full_risk_report.csv")

if not report_json_path.exists() or not risk_csv_path.exists():
    print("Error: Missing report files. Ensure you have run both previous notebooks.")
else:
    # Load JSON and convert to DataFrame to get the 'risk_flags' list per image
    with open(report_json_path, "r") as f:
        data1 = json.load(f)
    
    df1 = pd.DataFrame([{
        "image": r["image"],
        "base_flags": r["risk_flags"]  # List of strings like ['PHONE_DETECTED', 'BLURRY']
    } for r in data1])
    
    # Load CSV from notebook 2
    df2 = pd.read_csv(risk_csv_path)
    
    # Merge on image name
    df = df1.merge(df2, on="image", how="inner")
    print(f"Successfully merged data for {len(df)} images.")

## 2. Apply Category-Based Escalation Logic

In [ ]:
def compute_unified_risk(row):
    base_flags = row['base_flags'] if isinstance(row['base_flags'], list) else []
    
    cat_a = []
    cat_b = []
    cat_c = []
    cat_d = []
    
    # Parse signals from Notebook 1 (Base flags)
    if "PHONE_DETECTED" in base_flags: cat_a.append("Phone detected")
    
    if "MULTIPLE_FACES" in base_flags: cat_b.append("Multiple faces")
    
    if "BOOK_DETECTED" in base_flags: cat_c.append("Book detected")
    if "LAPTOP_DETECTED" in base_flags: cat_c.append("Laptop detected")
    if "MONITOR_DETECTED" in base_flags: cat_c.append("Monitor detected")
    if "FACE_MISSING" in base_flags: cat_c.append("Face missing")
    if "NO_PERSON" in base_flags: cat_c.append("No person detected")
    
    if "LOW_LIGHT" in base_flags: cat_d.append("Low light")
    if "OVEREXPOSED" in base_flags: cat_d.append("Overexposed")
    if "BLURRY" in base_flags: cat_d.append("Blurry")
    if "CAMERA_BLOCKED" in base_flags: cat_d.append("Camera blocked")
        
    # Parse signals from Notebook 2 (Extra person / Gaze)
    looking_toward = row.get('looking_toward_extra', False)
    extra_risk = row.get('base_risk_level', 'NONE')
    
    if looking_toward: cat_b.append("Looking toward extra person")
    if extra_risk in ["MEDIUM", "HIGH"]: cat_b.append(f"Extra person ({extra_risk})")
    
    if extra_risk == "LOW": cat_d.append("Extra person in background")
        
    # Escalation Rules
    if len(cat_a) > 0:
        return "HIGH", "CRITICAL: " + ", ".join(cat_a)
    
    if len(cat_b) >= 2:
        return "HIGH", "MULTIPLE SUSPICIOUS: " + ", ".join(cat_b)
    
    if len(cat_b) >= 1 and len(cat_c) >= 1:
        return "HIGH", "SUSPICIOUS + CONTEXTUAL: " + ", ".join(cat_b + cat_c)
    
    if len(cat_b) == 1:
        return "MEDIUM", "SUSPICIOUS: " + ", ".join(cat_b)
    
    if len(cat_c) >= 2:
        return "MEDIUM", "MULTIPLE CONTEXTUAL: " + ", ".join(cat_c)
    
    if len(cat_c) == 1:
        return "LOW", "MINOR CONTEXT: " + ", ".join(cat_c)
    
    if len(cat_d) > 0:
        return "LOW", "AMBIENT: " + ", ".join(cat_d)
    
    return "NONE", "Clean"

# Apply logic to dataframe
df[['unified_risk_level', 'unified_reason']] = df.apply(lambda row: pd.Series(compute_unified_risk(row)), axis=1)

## 3. View the Final Results

In [ ]:
from IPython.display import display
pd.set_option("display.max_colwidth", 80)
display_cols = ["image", "unified_risk_level", "unified_reason"]
display(df[display_cols].sort_values(by="unified_risk_level", key=lambda x: x.map({"HIGH": 3, "MEDIUM": 2, "LOW": 1, "NONE": 0}), ascending=False))

## 4. Save and Chart

In [ ]:
output_csv = Path("outputs/reports/unified_risk_report.csv")
df.to_csv(output_csv, index=False)
print(f"Saved final unified report to {output_csv}")

import matplotlib.pyplot as plt

counts = df["unified_risk_level"].value_counts()
color_map = {"NONE": "#9E9E9E", "LOW": "#FDD835", "MEDIUM": "#FB8C00", "HIGH": "#E53935"}
colors = [color_map.get(k, "gray") for k in counts.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(counts.index, counts.values, color=colors, edgecolor="white")
plt.title("Unified Risk Level Distribution")
plt.ylabel("Number of Images")

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.2, int(yval), ha="center", fontsize=10)

plt.show()